# Mario GA benchmark — Colab runner

Runs the same code as `mario_compare/` on your Mac: GA vs random search on
SMB 1-1, same frame budget, sticky actions 0.2, 30 held-out seeds.

**Runtime:** plain **CPU** is right. A GPU/TPU does not help — the NES
emulator runs on CPU and the 208→18→12 network is too small for a GPU.
Parallelism comes from running several seeds at once, one per CPU core.

Run the cells top to bottom. At the end, download `mario_results.zip`
and bring the printed summary table back.

## 1. Install (Python 3.13 via uv — `nes-py 9.0.1` needs ≥ 3.13)

In [ ]:
!pip -q install uv
!uv python install 3.13 -q
!uv venv --python 3.13 /content/venv -q
!uv pip install --python /content/venv/bin/python -q \
    nes-py==9.0.1 gym-super-mario-bros==9.1.0 gymnasium==1.3.0 numpy opencv-python-headless \
    cma stable-baselines3 torch
!/content/venv/bin/python -c "import nes_py, gym_super_mario_bros, gymnasium, numpy, cv2, cma, stable_baselines3; print('ok', numpy.__version__, gymnasium.__version__)"
!nproc

## 2. Write the project files

In [ ]:
import os
os.makedirs("/content/mario_compare/results", exist_ok=True)
for pkg in ["envs", "core", "policies", "optimizers", "experiments", "rl"]:
    os.makedirs(f"/content/mario_compare/{pkg}", exist_ok=True)
    open(f"/content/mario_compare/{pkg}/__init__.py", "w").close()
%cd /content/mario_compare

In [ ]:
%%writefile /content/mario_compare/envs/mario_env.py
"""
Shared Mario environment for the comparative study.

Every optimizer -- GA, ES, CMA-ES, PPO, random search, scripted baseline --
talks to THIS object and nothing else. Same observation, same action space,
same fitness, same episode termination rules. That is what makes the
comparison valid.

Observation : Box(-1, 2, shape=(208,))   flattened 16x13 tile grid  (obs_mode="grid")
              optional extras, same for every method (see OBS_MODES):
                "act"  + one-hot of the action actually executed last step  (220)
                "prev" + the previous step's grid, i.e. motion              (416)
                "both" + both                                               (428)
Action      : Discrete(12)               COMPLEX_MOVEMENT
Reward      : our own, derived from info['x_pos_max'] (NOT the env's reward)
"""

from __future__ import annotations

import numpy as np
import gymnasium as gym
from gymnasium import spaces

# ----------------------------------------------------------------------------
# Grid geometry
# ----------------------------------------------------------------------------
GRID_W, GRID_H = 16, 13
OBS_SIZE = GRID_W * GRID_H          # 208
N_ACTIONS = 12

OBS_MODES = ("grid", "act", "prev", "both")


def obs_size(mode: str) -> int:
    return (OBS_SIZE
            + (OBS_SIZE if mode in ("prev", "both") else 0)
            + (N_ACTIONS if mode in ("act", "both") else 0))

EMPTY, SOLID, MARIO, ENEMY = 0.0, 1.0, 2.0, -1.0

# ----------------------------------------------------------------------------
# SMB ROM RAM map. These are properties of the game cartridge, not of whatever
# python wrapper you have installed, so they are stable across forks.
# ----------------------------------------------------------------------------
TILE_BASE      = 0x500   # two-page ring buffer, each page 16 wide x 13 tall
ENEMY_DRAWN    = 0x0F    # 5 slots: 0x0F .. 0x13, nonzero = enemy active
ENEMY_X_PAGE   = 0x6E    # 5 slots
ENEMY_X_SCREEN = 0x87    # 5 slots
ENEMY_Y_SCREEN = 0xCF    # 5 slots

# Calibration constants. Standing on flat ground, Mario and a walking Goomba
# should both land on row 10 -- directly on top of the floor at rows 11-12.
MARIO_Y_OFFSET = -16
ENEMY_Y_OFFSET = -24


def tile_address(tile_x: int, row: int) -> int:
    """Absolute tile column + row -> RAM address in the 0x500 ring buffer."""
    page = (tile_x // 16) % 2        # %2 because the buffer wraps every 2 pages
    return TILE_BASE + (tile_x % 16) + (page * 13 + row) * 16


def default_factory(level: str = "SuperMarioBros-1-1-v0"):
    """Builds the raw env. Edit here if your fork uses different imports."""
    import gym_super_mario_bros
    from nes_py.wrappers import JoypadSpace
    from gym_super_mario_bros.actions import COMPLEX_MOVEMENT
    return JoypadSpace(gym_super_mario_bros.make(level), COMPLEX_MOVEMENT)


class MarioTileEnv(gym.Env):
    """Gymnasium env exposing the 208-dim tile grid."""

    metadata = {"render_modes": ["rgb_array"]}

    def __init__(
        self,
        level: str = "SuperMarioBros-1-1-v0",
        frame_skip: int = 4,
        max_noop: int = 30,
        stall_limit: int = 240,      # frames without progress before truncating
        death_penalty: float = 15.0,
        flag_bonus: float = 1000.0,
        sticky_prob: float = 0.2,    # Machado et al. 2018 sticky actions
        obs_mode: str = "grid",
        factory=default_factory,
    ):
        super().__init__()
        self.env = factory(level)
        self.frame_skip = frame_skip
        self.max_noop = max_noop
        self.stall_limit = stall_limit
        self.death_penalty = death_penalty
        self.flag_bonus = flag_bonus
        self.sticky_prob = sticky_prob
        assert obs_mode in OBS_MODES, f"obs_mode must be one of {OBS_MODES}"
        self.obs_mode = obs_mode

        self.observation_space = spaces.Box(
            low=-1.0, high=2.0, shape=(obs_size(obs_mode),), dtype=np.float32
        )
        self.action_space = spaces.Discrete(N_ACTIONS)

        # bookkeeping used for fitness + stall detection
        self._max_x = 0
        self._stall = 0
        self._frames = 0            # raw emulator frames consumed this episode
        self._last_frame = None
        self._prev_action = 0       # action actually executed last step
        self._prev_grid = None      # last step's grid, for obs_mode prev/both

    # ------------------------------------------------------------------
    # observation
    # ------------------------------------------------------------------
    @property
    def ram(self):
        return self.env.unwrapped.ram

    def _build_grid(self, info: dict) -> np.ndarray:
        ram = self.ram
        grid = np.zeros((GRID_H, GRID_W), dtype=np.float32)

        # world-pixel x of screen column 0, straight from info
        left = int(info.get("left_x_pos", 0))

        # --- solid tiles -------------------------------------------------
        for col in range(GRID_W):
            tile_x = (left + col * 16) // 16
            for row in range(GRID_H):
                if ram[tile_address(tile_x, row)] != 0:
                    grid[row, col] = SOLID

        # --- mario -------------------------------------------------------
        m_col = (int(info.get("x_pos", 0)) - left) // 16
        m_row = (int(info.get("y_pixel", 0)) + MARIO_Y_OFFSET) // 16
        if 0 <= m_row < GRID_H and 0 <= m_col < GRID_W:
            grid[m_row, m_col] = MARIO

        # --- enemies (5 hardware slots) ----------------------------------
        for i in range(5):
            if ram[ENEMY_DRAWN + i] == 0:
                continue
            ex = int(ram[ENEMY_X_PAGE + i]) * 256 + int(ram[ENEMY_X_SCREEN + i])
            ey = int(ram[ENEMY_Y_SCREEN + i])
            e_col = (ex - left) // 16
            e_row = (ey + ENEMY_Y_OFFSET) // 16
            if 0 <= e_row < GRID_H and 0 <= e_col < GRID_W:
                grid[e_row, e_col] = ENEMY

        return grid.flatten()

    def _observe(self, info: dict) -> np.ndarray:
        grid = self._build_grid(info)
        if self.obs_mode == "grid":
            return grid
        parts = [grid]
        if self.obs_mode in ("prev", "both"):
            # first step of an episode: previous = current, i.e. "no motion"
            parts.append(grid if self._prev_grid is None else self._prev_grid)
            self._prev_grid = grid
        if self.obs_mode in ("act", "both"):
            onehot = np.zeros(N_ACTIONS, dtype=np.float32)
            onehot[self._prev_action] = 1.0
            parts.append(onehot)
        return np.concatenate(parts)

    # ------------------------------------------------------------------
    # gym api
    # ------------------------------------------------------------------
    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        _, info = self.env.reset(seed=seed)

        # Random no-op start. SMB is fully deterministic; without this a GA
        # can evolve a memorised button tape instead of a policy.
        # Deliberately bypasses sticky logic: prev action is 0 and the request
        # is 0, so stickiness would only burn RNG draws for no effect.
        n = int(self.np_random.integers(0, self.max_noop + 1))
        for _ in range(n):
            _, _, term, trunc, info = self.env.step(0)
            if term or trunc:
                _, info = self.env.reset()
                break

        self._max_x = int(info.get("x_pos_max", 0))
        self._stall = 0
        self._frames = n
        self._prev_action = 0
        self._prev_grid = None
        obs = self._observe(info)
        return obs, self._pack(info)

    def step(self, action: int):
        total_env_reward = 0.0
        terminated = truncated = False
        info = {}

        # Sticky actions: one coin flip per agent decision, not per frame.
        # Uses self.np_random so the stream is reseeded by reset(seed=...).
        action = int(action)
        if self.np_random.random() < self.sticky_prob:
            action = self._prev_action
        self._prev_action = action

        for _ in range(self.frame_skip):
            _, r, terminated, truncated, info = self.env.step(int(action))
            total_env_reward += float(r)
            self._frames += 1
            if terminated or truncated:
                break

        # ---- our own reward, independent of the fork's shaped/clipped one
        x_max = int(info.get("x_pos_max", self._max_x))
        gained = max(0, x_max - self._max_x)
        self._max_x = x_max

        reward = float(gained)
        self._stall = 0 if gained > 0 else self._stall + self.frame_skip

        if info.get("flag_get"):
            reward += self.flag_bonus
            terminated = True
        if info.get("is_dead") or info.get("is_dying") or info.get("is_game_over"):
            reward -= self.death_penalty
            terminated = True
        if self._stall >= self.stall_limit:
            truncated = True

        obs = self._observe(info)
        return obs, reward, terminated, truncated, self._pack(info, total_env_reward)

    def _pack(self, info: dict, env_reward: float = 0.0) -> dict:
        out = dict(info)
        out["max_x"] = self._max_x
        out["frames"] = self._frames          # budget accounting lives here
        out["env_reward"] = env_reward        # kept for reference, never used
        return out

    def render(self):
        return self.env.render()

    def close(self):
        self.env.close()


# ----------------------------------------------------------------------------
# Step 2b: grid calibration. Run this file directly.
# ----------------------------------------------------------------------------
SYMBOLS = {0.0: ".", 1.0: "#", 2.0: "M", -1.0: "E"}

def ascii_grid(obs: np.ndarray) -> str:
    g = obs.reshape(GRID_H, GRID_W)
    return "\n".join("".join(SYMBOLS[float(v)] for v in row) for row in g)


if __name__ == "__main__":
    env = MarioTileEnv(frame_skip=4, max_noop=0)
    obs, info = env.reset(seed=0)

    print("obs space:", env.observation_space)
    print("first info keys:", sorted(info.keys())[:8], "...")

    for t in range(120):
        obs, r, term, trunc, info = env.step(1)   # 1 = right
        if t % 20 == 0:
            ram = env.ram
            enemies = [
                (i, int(ram[ENEMY_X_PAGE + i]) * 256 + int(ram[ENEMY_X_SCREEN + i]),
                 int(ram[ENEMY_Y_SCREEN + i]))
                for i in range(5) if ram[ENEMY_DRAWN + i] != 0
            ]
            print(f"\n--- t={t} x_max={info['max_x']} frames={info['frames']} r={r}")
            print(f"    x_pos={info.get('x_pos')} left_x_pos={info.get('left_x_pos')} "
                  f"y_pixel={info.get('y_pixel')} y_pos={info.get('y_pos')} "
                  f"screen_col={(int(info.get('x_pos',0))-int(info.get('left_x_pos',0)))//16}")
            print(f"    enemies(slot,world_x,screen_y)={enemies}")
            print(ascii_grid(obs))
        if term or trunc:
            print("episode ended at t =", t, "reason:",
                  "terminated" if term else "truncated")
            break
    env.close()

In [ ]:
%%writefile /content/mario_compare/core/rollout.py
"""
Shared evaluation protocol.

Every optimizer in the study calls evaluate() and nothing else. This is where
fairness is enforced:

  * the same seed set for every method within a generation/iteration
  * every consumed emulator frame charged to a single shared budget
  * identical episode termination rules (they live in the env)

If an optimizer ever steps the env directly, the comparison is broken.
"""

from __future__ import annotations
from dataclasses import dataclass, field
import numpy as np


class BudgetExhausted(Exception):
    """Raised when the shared frame budget runs out mid-search."""


@dataclass
class FrameBudget:
    """The experimental currency. Every method gets the same total."""
    total: int
    used: int = 0

    def charge(self, frames: int) -> None:
        self.used += frames
        if self.used >= self.total:
            raise BudgetExhausted(f"{self.used}/{self.total} frames")

    @property
    def remaining(self) -> int:
        return max(0, self.total - self.used)

    @property
    def fraction(self) -> float:
        return self.used / self.total


@dataclass
class RolloutResult:
    fitness: float
    max_x: int
    frames: int
    flag: bool
    env_reward: float
    final_x: int = 0        # where the episode ended -- the "behaviour" used by
    final_y: int = 0        # novelty search and MAP-Elites. Nothing else reads it.


def rollout(env, policy, seed: int, max_steps: int = 3000) -> RolloutResult:
    """One episode. Returns our fitness, not the env's reward."""
    obs, info = env.reset(seed=seed)
    policy.reset()

    start_frames = info["frames"]
    fitness = 0.0
    env_reward = 0.0

    for _ in range(max_steps):
        obs, r, terminated, truncated, info = env.step(policy.act(obs))
        fitness += r
        env_reward += info.get("env_reward", 0.0)
        if terminated or truncated:
            break

    return RolloutResult(
        fitness=fitness,
        max_x=int(info.get("max_x", 0)),
        frames=int(info["frames"] - start_frames),
        flag=bool(info.get("flag_get", False)),
        env_reward=env_reward,
        final_x=int(info.get("x_pos", 0)),
        final_y=int(info.get("y_pos", 0)),
    )


def evaluate(env, policy, seeds, budget: FrameBudget | None = None) -> dict:
    """Average a policy over a fixed seed set.

    Noop starts make every episode stochastic, so a single rollout is a noisy
    estimate of a policy's quality. More seeds = less noise but proportionally
    more frames spent. That trade-off is itself worth a plot.
    """
    results = [rollout(env, policy, int(s)) for s in seeds]
    frames = sum(r.frames for r in results)
    if budget is not None:
        budget.charge(frames)

    fits = np.array([r.fitness for r in results], dtype=np.float64)
    return {
        "fitness": float(fits.mean()),
        "fitness_std": float(fits.std()),
        "fitness_min": float(fits.min()),
        "fitness_max": float(fits.max()),
        "max_x": float(np.mean([r.max_x for r in results])),
        "best_x": int(max(r.max_x for r in results)),
        "flag_rate": float(np.mean([r.flag for r in results])),
        "env_reward": float(np.mean([r.env_reward for r in results])),
        "frames": frames,
        "final_x": float(np.mean([r.final_x for r in results])),
        "final_y": float(np.mean([r.final_y for r in results])),
    }


def seed_set(rng: np.random.Generator, n: int, hi: int = 10**6) -> list[int]:
    """Fresh seeds for a search iteration (prevents overfitting to fixed starts)."""
    return [int(s) for s in rng.integers(0, hi, size=n)]


# Validation seeds: for choosing settings (sigma, crossover, ...) between runs.
# Never used during search, never used for final numbers.
VAL_SEEDS = list(range(800_000, 800_030))

# Held-out seeds, never used during search or for choosing settings.
# Final reported numbers use these, once, for the chosen setting only.
TEST_SEEDS = list(range(900_000, 900_030))

In [ ]:
%%writefile /content/mario_compare/policies/mlp.py
"""
Policy layer.

The contract every optimizer relies on:

    policy.act(obs) -> int in [0, 12)
    policy.get_params() -> flat float64 array
    policy.set_params(vec) -> None
    policy.n_params -> int

Because parameters are a flat vector, GA / ES / CMA-ES / random search are all
just different ways of proposing vectors. None of them needs to know that the
vector happens to describe a 208-18-12 network.
"""

from __future__ import annotations
import numpy as np

N_INPUTS, N_HIDDEN, N_OUTPUTS = 208, 18, 12

# COMPLEX_MOVEMENT indices (12 actions):
#  0 NOOP        1 right       2 right+A      3 right+B
#  4 right+A+B   5 A           6 left         7 left+A
#  8 left+B      9 left+A+B   10 down        11 up
RIGHT, RIGHT_JUMP, RIGHT_RUN, RIGHT_RUNJUMP = 1, 2, 3, 4


class Policy:
    """Base class. Stateless by default; reset() exists for recurrent policies."""
    n_params = 0

    def act(self, obs: np.ndarray) -> int:
        raise NotImplementedError

    def reset(self) -> None:
        pass

    def get_params(self) -> np.ndarray:
        return np.zeros(0)

    def set_params(self, vec: np.ndarray) -> None:
        pass


# ---------------------------------------------------------------------------
# The evolvable network -- same topology as the original repo
# ---------------------------------------------------------------------------
class MLPPolicy(Policy):
    """208 -> 18 (ReLU) -> 12, argmax. 3990 parameters."""

    def __init__(self, hidden: int = N_HIDDEN, init_scale: float = 0.1, rng=None,
                 n_inputs: int = N_INPUTS):
        self.hidden = hidden
        self.n_inputs = n_inputs
        self.rng = rng or np.random.default_rng()
        self.shapes = [
            (n_inputs, hidden),   # W1
            (hidden,),            # b1
            (hidden, N_OUTPUTS),  # W2
            (N_OUTPUTS,),         # b2
        ]
        self.sizes = [int(np.prod(s)) for s in self.shapes]
        self.n_params = int(sum(self.sizes))
        self.set_params(self._random_init(init_scale))

    def _random_init(self, scale: float) -> np.ndarray:
        parts = []
        for shape in self.shapes:
            if len(shape) == 2:              # weights
                parts.append(scale * self.rng.standard_normal(shape).ravel())
            else:                            # biases start at zero
                parts.append(np.zeros(int(np.prod(shape))))
        return np.concatenate(parts)

    # -- flat vector <-> matrices -------------------------------------------
    def set_params(self, vec: np.ndarray) -> None:
        vec = np.asarray(vec, dtype=np.float64).ravel()
        assert vec.size == self.n_params, f"expected {self.n_params}, got {vec.size}"
        self._flat = vec.copy()
        out, i = [], 0
        for shape, size in zip(self.shapes, self.sizes):
            out.append(self._flat[i:i + size].reshape(shape))
            i += size
        self.W1, self.b1, self.W2, self.b2 = out

    def get_params(self) -> np.ndarray:
        return self._flat.copy()

    # -- forward pass --------------------------------------------------------
    def act(self, obs: np.ndarray) -> int:
        h = np.maximum(0.0, obs @ self.W1 + self.b1)     # ReLU
        return int(np.argmax(h @ self.W2 + self.b2))     # argmax


# ---------------------------------------------------------------------------
# Baselines -- the two numbers every other method must beat
# ---------------------------------------------------------------------------
class HoldRightPolicy(Policy):
    """Scripted: run right, jump on a fixed duty cycle. Zero parameters.

    This is the floor. Any method that does not clearly beat it has learned
    nothing, no matter how good its fitness curve looks in isolation.
    """

    def __init__(self, jump_period: int = 0, jump_len: int = 6):
        self.jump_period = jump_period    # 0 = never jump
        self.jump_len = jump_len
        self.t = 0

    def reset(self) -> None:
        self.t = 0

    def act(self, obs: np.ndarray) -> int:
        self.t += 1
        if self.jump_period and (self.t % self.jump_period) < self.jump_len:
            return RIGHT_RUNJUMP
        return RIGHT_RUN


class RandomPolicy(Policy):
    """Uniform random actions. The absolute floor -- expect near zero."""

    def __init__(self, seed: int = 0):
        self.rng = np.random.default_rng(seed)

    def act(self, obs: np.ndarray) -> int:
        return int(self.rng.integers(0, N_OUTPUTS))


if __name__ == "__main__":
    p = MLPPolicy()
    print("parameters:", p.n_params)          # 3990
    obs = np.zeros(N_INPUTS, dtype=np.float32)
    print("action on blank obs:", p.act(obs))
    v = p.get_params()
    p.set_params(v)                            # round-trip must be lossless
    assert np.allclose(v, p.get_params())
    print("round-trip ok")

In [ ]:
%%writefile /content/mario_compare/optimizers/ga.py
"""
Genetic algorithm over flat MLP weight vectors.

This is run_random_search plus three things: selection, mutation, elitism
(crossover optional). Anything the GA beats random search by is what those
operators contribute.
"""

from __future__ import annotations
import numpy as np

from policies.mlp import MLPPolicy
from core.rollout import BudgetExhausted,evaluate,seed_set,TEST_SEEDS



## Selection and crossover
def tournament(rng,fits:np.ndarray,k:int):
    # pick k random individuals , return the index of the fittest

    idx = rng.choice(len(fits),size = k, replace = False)
    return int(idx[np.argmax(fits[idx])])


def uniform_crossover(rng, a: np.ndarray,b : np.ndarray):
    #Each gene comes from a or b with probability 1/2.
    return np.where(rng.random(a.size)<0.5,a,b)


## Run the main loop

def run_ga(env, budget, log, pop_size = 24, n_elite=2, k = 3 , sigma = 0.03,n_seeds = 3, crossover = False, seed = 0, stall_gens = 0):
    rng = np.random.default_rng(seed)
    policy = MLPPolicy(rng=rng, n_inputs=env.observation_space.shape[0])

    n = policy.n_params

    # same init distribution as random search, so generation 0 is identical to it
    pop = [0.1* rng.standard_normal(n) for _ in range(pop_size)]
    best, gen = pop[0],0
    # adaptive mutation: sig grows while the best score is stuck (stall_gens=0 -> off)
    best_ever, stall, sig = -np.inf, 0, sigma


    try:
        while True:

            seeds = seed_set(rng,n_seeds)
            fits = np.empty(pop_size)
            for i, genome in enumerate(pop):
                policy.set_params(genome)
                fits[i] = evaluate(env,policy,seeds,budget)["fitness"]
            
            order = np.argsort(fits)[::-1]

            best = pop[order[0]]

            gen+=1
            log(dict(method="ga", iteration=gen, frames_used=budget.used,
                     fitness=fits.mean(), fitness_std=fits.std(),
                     fitness_min=fits.min(), fitness_max=fits.max()))
            print(f"  [ga] gen={gen:3d} best={fits.max():7.1f} "
                  f"mean={fits.mean():7.1f}  budget={budget.fraction:.1%}")

            if fits.max() > best_ever + 1:
                best_ever, stall, sig = fits.max(), 0, sigma
            elif stall_gens:
                stall += 1
                if stall >= stall_gens:
                    sig = min(sig * 1.5, 4 * sigma)

            children = [pop[i].copy() for i in order[:n_elite]]
            while len(children) < pop_size:
                child = pop[tournament(rng, fits, k)].copy()
                if crossover:
                    other = pop[tournament(rng, fits, k)]
                    child = uniform_crossover(rng, child, other)
                child += sig * rng.standard_normal(n)
                children.append(child)
            pop = children
    except BudgetExhausted:
        pass    # the generation that ran out is discarded; best is from the last full one

    policy.set_params(best)
    final = evaluate(env, policy, TEST_SEEDS)      # held-out, not charged
    print(f"  [ga] {gen} generations, held-out: fitness={final['fitness']:.1f} "
          f"x={final['max_x']:.1f}")
    return best, final, gen

In [ ]:
%%writefile /content/mario_compare/optimizers/es.py
"""
OpenAI-style Evolution Strategies (Salimans et al. 2017) over flat MLP weights.

The GA keeps a population of competing individuals. ES keeps ONE weight vector
theta and estimates which direction improves it:

    1. perturb theta with Gaussian noise, in +/- pairs (antithetic sampling)
    2. score every perturbation on the same seeds
    3. move theta toward the noise directions that scored well

Same evaluate(), same frame budget, same seeds protocol as the GA -- only the
way new weight vectors are proposed differs.
"""

from __future__ import annotations
import numpy as np

from policies.mlp import MLPPolicy
from core.rollout import BudgetExhausted, evaluate, seed_set, TEST_SEEDS


def centered_ranks(x: np.ndarray, rng=None) -> np.ndarray:
    """Replace scores by their rank, scaled to [-0.5, 0.5].

    Only the ordering matters, so one lucky 900-point episode cannot drag
    the whole update toward it.

    Ties (common: every candidate stuck at the same pipe scores the same):
      rng given -> broken in random order. ES uses this: on a plateau the
                   update becomes a random step, so theta random-walks until
                   something scores differently. (The reference implementation
                   gets the same effect from argsort's arbitrary order.) With
                   tied ranks averaged instead, a run whose every candidate
                   scored the same never moved again.
      rng None  -> tied scores share their average rank (used by novelty
                   search, where ties should carry no preference).
    """
    if rng is not None:
        order = np.lexsort((rng.random(len(x)), x))     # sort by x, ties by a random key
        ranks = np.empty(len(x))
        ranks[order] = np.arange(len(x))
    else:
        ranks = np.empty(len(x))
        ranks[np.argsort(x, kind="stable")] = np.arange(len(x))
        _, groups = np.unique(x, return_inverse=True)
        ranks = (np.bincount(groups, ranks) / np.bincount(groups))[groups]
    return ranks / (len(x) - 1) - 0.5


class Adam:
    """Standard Adam, used for gradient ASCENT (the step is added to theta)."""

    def __init__(self, n: int, lr: float, b1=0.9, b2=0.999, eps=1e-8):
        self.lr, self.b1, self.b2, self.eps = lr, b1, b2, eps
        self.m, self.v, self.t = np.zeros(n), np.zeros(n), 0

    def step(self, grad: np.ndarray) -> np.ndarray:
        self.t += 1
        self.m = self.b1 * self.m + (1 - self.b1) * grad
        self.v = self.b2 * self.v + (1 - self.b2) * grad * grad
        m_hat = self.m / (1 - self.b1 ** self.t)
        v_hat = self.v / (1 - self.b2 ** self.t)
        return self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


def run_es(env, budget, log, n_pairs=12, sigma=0.02, lr=0.01, weight_decay=0.005,
           n_seeds=3, seed=0):
    rng = np.random.default_rng(seed)
    policy = MLPPolicy(rng=rng, n_inputs=env.observation_space.shape[0])
    n = policy.n_params

    # same init distribution as GA / random search
    theta = 0.1 * rng.standard_normal(n)
    opt = Adam(n, lr)
    it = 0

    try:
        while True:
            # one seed set per iteration, shared by every perturbation (fair ranking)
            seeds = seed_set(rng, n_seeds)
            noise = rng.standard_normal((n_pairs, n))
            fits = np.empty(2 * n_pairs)
            for i in range(n_pairs):
                for j, sign in enumerate((1.0, -1.0)):
                    policy.set_params(theta + sign * sigma * noise[i])
                    fits[2 * i + j] = evaluate(env, policy, seeds, budget)["fitness"]

            # antithetic estimate: for each pair, (+ rank) - (- rank) says which way is uphill
            w = centered_ranks(fits, rng)
            w = w[0::2] - w[1::2]
            grad = w @ noise / (2 * n_pairs * sigma)
            # weight decay applied outside Adam (AdamW-style): inside, Adam would rescale
            # it to a full lr-sized step whenever the rank gradient is zero (all ties)
            theta = theta + opt.step(grad) - lr * weight_decay * theta

            it += 1
            log(dict(method="es", iteration=it, frames_used=budget.used,
                     fitness=fits.mean(), fitness_std=fits.std(),
                     fitness_min=fits.min(), fitness_max=fits.max()))
            print(f"  [es] it={it:3d} best={fits.max():7.1f} "
                  f"mean={fits.mean():7.1f}  budget={budget.fraction:.1%}")
    except BudgetExhausted:
        pass    # the iteration that ran out is discarded; theta is from the last full update

    policy.set_params(theta)
    final = evaluate(env, policy, TEST_SEEDS)      # held-out, not charged
    print(f"  [es] {it} iterations, held-out: fitness={final['fitness']:.1f} "
          f"x={final['max_x']:.1f}")
    return theta, final, it

In [ ]:
%%writefile /content/mario_compare/optimizers/hill_climber.py
"""
(1+1) hill climber over flat MLP weights.

The simplest search that is smarter than random search: keep ONE network,
propose one mutated copy, keep whichever scores better. No population, no
crossover, no gradient estimate.

Its place in the benchmark: GA minus the population. If the GA barely beats
this, the population is not what makes the GA work.
"""

from __future__ import annotations
import numpy as np

from policies.mlp import MLPPolicy
from core.rollout import BudgetExhausted, evaluate, seed_set


def run_hill_climber(env, budget, log, sigma=0.05, n_seeds=3, seed=0):
    rng = np.random.default_rng(seed)
    policy = MLPPolicy(rng=rng, n_inputs=env.observation_space.shape[0])
    n = policy.n_params

    parent = 0.1 * rng.standard_normal(n)       # same init as every other method
    it, accepted = 0, 0

    try:
        while True:
            # parent and child are scored on the SAME fresh seeds every iteration.
            # Re-scoring the parent costs frames, but otherwise a parent that got
            # lucky once would block every child forever.
            seeds = seed_set(rng, n_seeds)
            child = parent + sigma * rng.standard_normal(n)

            policy.set_params(parent)
            f_parent = evaluate(env, policy, seeds, budget)["fitness"]
            policy.set_params(child)
            f_child = evaluate(env, policy, seeds, budget)["fitness"]

            # >= (not >): accepting equal children lets the climber drift across
            # flat regions, e.g. while every variant is stuck at the same pipe
            if f_child >= f_parent:
                parent, accepted = child, accepted + 1

            it += 1
            log(dict(method="hc", iteration=it, frames_used=budget.used,
                     fitness=max(f_parent, f_child), fitness_std=abs(f_child - f_parent) / 2,
                     fitness_min=min(f_parent, f_child), fitness_max=max(f_parent, f_child)))
            if it % 20 == 0:
                print(f"  [hc] it={it:4d} parent={f_parent:7.1f} child={f_child:7.1f} "
                      f"accepted={accepted / it:.0%}  budget={budget.fraction:.1%}")
    except BudgetExhausted:
        pass

    return parent, it

In [ ]:
%%writefile /content/mario_compare/optimizers/cem.py
"""
Cross-Entropy Method (CEM) over flat MLP weights.

Keeps a Gaussian over weight vectors (a mean and a per-weight std). Each
iteration:

    1. sample a population from the Gaussian
    2. score everyone on the same seeds
    3. refit the Gaussian to the top `elite_frac` of them

The distribution slides toward good regions and narrows as it gains
confidence. Extra noise is added to the std (Szita & Lorincz 2006) so it cannot
collapse to zero before it has found anything.
"""

from __future__ import annotations
import numpy as np

from policies.mlp import MLPPolicy
from core.rollout import BudgetExhausted, evaluate, seed_set


def run_cem(env, budget, log, pop_size=24, elite_frac=0.2, init_std=0.1,
            extra_std=0.02, n_seeds=3, seed=0):
    rng = np.random.default_rng(seed)
    policy = MLPPolicy(rng=rng, n_inputs=env.observation_space.shape[0])
    n = policy.n_params
    n_elite = max(2, int(round(pop_size * elite_frac)))

    # mean 0, std 0.1: generation 0 has the same distribution as every other method's init
    mean = np.zeros(n)
    std = np.full(n, init_std)
    it = 0

    try:
        while True:
            seeds = seed_set(rng, n_seeds)
            samples = mean + std * rng.standard_normal((pop_size, n))
            fits = np.empty(pop_size)
            for i, x in enumerate(samples):
                policy.set_params(x)
                fits[i] = evaluate(env, policy, seeds, budget)["fitness"]

            elites = samples[np.argsort(fits)[::-1][:n_elite]]
            mean = elites.mean(axis=0)
            std = np.sqrt(elites.var(axis=0) + extra_std ** 2)

            it += 1
            log(dict(method="cem", iteration=it, frames_used=budget.used,
                     fitness=fits.mean(), fitness_std=fits.std(),
                     fitness_min=fits.min(), fitness_max=fits.max()))
            print(f"  [cem] it={it:3d} best={fits.max():7.1f} mean={fits.mean():7.1f} "
                  f"avg std={std.mean():.4f}  budget={budget.fraction:.1%}")
    except BudgetExhausted:
        pass    # the iteration that ran out is discarded

    # the answer is the distribution's mean, not any single sample
    return mean, it

In [ ]:
%%writefile /content/mario_compare/optimizers/cmaes.py
"""
CMA-ES over flat MLP weights, via the `cma` package (Hansen's reference implementation).

Like CEM it samples from a Gaussian and moves it toward the better samples,
but it also:
  * learns the SHAPE of the Gaussian (which directions are worth searching),
  * adapts the overall step size sigma from how consistently the mean moves,
  * weights every ranked sample instead of a hard top-20% cut.

Full CMA-ES stores a 3990 x 3990 covariance matrix and decomposes it every few
iterations -- too slow and too data-hungry for ~45 iterations. The default here
is sep-CMA-ES (Ros & Hansen 2008): a diagonal covariance, i.e. one learned
step size per weight. --full switches to the full matrix.
"""

from __future__ import annotations
import numpy as np
import cma

from policies.mlp import MLPPolicy
from core.rollout import BudgetExhausted, evaluate, seed_set


def run_cmaes(env, budget, log, pop_size=24, sigma0=0.1, diagonal=True, n_seeds=3, seed=0):
    rng = np.random.default_rng(seed)
    policy = MLPPolicy(rng=rng, n_inputs=env.observation_space.shape[0])
    n = policy.n_params

    es = cma.CMAEvolutionStrategy(np.zeros(n), sigma0, {
        "popsize": pop_size,
        "seed": seed + 1,            # cma treats seed 0 as "random"; +1 keeps runs reproducible
        "CMA_diagonal": diagonal,
        "verbose": -9,
        # never stop early: the frame budget is the only stopping rule, as for every method
        "tolfun": 0, "tolx": 0, "tolfunhist": 0, "tolflatfitness": np.inf,
        "tolstagnation": np.inf, "maxiter": np.inf,
    })
    it = 0

    try:
        while True:
            seeds = seed_set(rng, n_seeds)
            xs = es.ask()
            fits = np.empty(pop_size)
            for i, x in enumerate(xs):
                policy.set_params(x)
                fits[i] = evaluate(env, policy, seeds, budget)["fitness"]
            es.tell(xs, (-fits).tolist())     # cma minimises, we maximise

            it += 1
            log(dict(method="cmaes", iteration=it, frames_used=budget.used,
                     fitness=fits.mean(), fitness_std=fits.std(),
                     fitness_min=fits.min(), fitness_max=fits.max()))
            print(f"  [cmaes] it={it:3d} best={fits.max():7.1f} mean={fits.mean():7.1f} "
                  f"sigma={es.sigma:.4f}  budget={budget.fraction:.1%}")
    except BudgetExhausted:
        pass    # the iteration that ran out is never told to cma

    # the distribution mean is CMA-ES's best estimate of the optimum
    return np.array(es.mean), it

In [ ]:
%%writefile /content/mario_compare/optimizers/novelty.py
"""
Novelty search, combined with fitness, on top of the GA.

The GA only rewards "got further right". When the whole population is stuck at
the same pipe, they all score the same and selection has nothing to work with.
Novelty search also rewards behaving DIFFERENTLY from what has been seen before
(Lehman & Stanley 2011): ending on top of a pipe, or in a new spot, counts for
something even if it is not yet further right.

Behaviour  = where the episode ended (final x, final y), in tiles, averaged
             over the evaluation seeds.
Novelty    = mean distance to the k nearest behaviours among the current
             population and an archive of past behaviours.
Selection  = tournament on a blend of novelty rank and fitness rank
             (novelty_weight=1 is pure novelty search, 0 is the plain GA).
Elites     = still kept by FITNESS, and the returned network is the fittest,
             so the benchmark measures the same thing as for every method.
"""

from __future__ import annotations
import numpy as np

from policies.mlp import MLPPolicy
from core.rollout import BudgetExhausted, evaluate, seed_set
from optimizers.ga import tournament, uniform_crossover
from optimizers.es import centered_ranks

TILE = 16.0


def behaviour(res: dict) -> np.ndarray:
    return np.array([res["final_x"] / TILE, res["final_y"] / TILE])


def novelty_scores(bcs: np.ndarray, archive: list, k: int) -> np.ndarray:
    """Mean distance from each behaviour to its k nearest neighbours
    (other population members + archive; never itself)."""
    ref = np.vstack([bcs] + ([np.array(archive)] if archive else []))
    d = np.linalg.norm(bcs[:, None, :] - ref[None, :, :], axis=2)
    d[np.arange(len(bcs)), np.arange(len(bcs))] = np.inf      # exclude self
    k = min(k, ref.shape[0] - 1)
    return np.sort(d, axis=1)[:, :k].mean(axis=1)


def run_novelty(env, budget, log, pop_size=24, n_elite=2, k=3, sigma=0.05,
                crossover=True, novelty_weight=0.5, k_nn=10, archive_prob=0.1,
                n_seeds=3, seed=0):
    rng = np.random.default_rng(seed)
    policy = MLPPolicy(rng=rng, n_inputs=env.observation_space.shape[0])
    n = policy.n_params

    pop = [0.1 * rng.standard_normal(n) for _ in range(pop_size)]
    archive: list[np.ndarray] = []
    best, gen = pop[0], 0

    try:
        while True:
            seeds = seed_set(rng, n_seeds)
            fits = np.empty(pop_size)
            bcs = np.empty((pop_size, 2))
            for i, genome in enumerate(pop):
                policy.set_params(genome)
                res = evaluate(env, policy, seeds, budget)
                fits[i], bcs[i] = res["fitness"], behaviour(res)

            nov = novelty_scores(bcs, archive, k_nn)
            score = novelty_weight * centered_ranks(nov) + (1 - novelty_weight) * centered_ranks(fits)

            # archive: a random sample of behaviours seen so far (Lehman & Stanley)
            for b in bcs[rng.random(pop_size) < archive_prob]:
                archive.append(b)

            order = np.argsort(fits)[::-1]
            best = pop[order[0]]
            gen += 1
            log(dict(method="novelty", iteration=gen, frames_used=budget.used,
                     fitness=fits.mean(), fitness_std=fits.std(),
                     fitness_min=fits.min(), fitness_max=fits.max()))
            print(f"  [novelty] gen={gen:3d} best={fits.max():7.1f} mean={fits.mean():7.1f} "
                  f"novelty={nov.mean():5.2f} archive={len(archive):3d}  budget={budget.fraction:.1%}")

            children = [pop[i].copy() for i in order[:n_elite]]     # elites by fitness
            while len(children) < pop_size:
                child = pop[tournament(rng, score, k)].copy()       # parents by blended score
                if crossover:
                    child = uniform_crossover(rng, child, pop[tournament(rng, score, k)])
                child += sigma * rng.standard_normal(n)
                children.append(child)
            pop = children
    except BudgetExhausted:
        pass    # the generation that ran out is discarded; best is from the last full one

    return best, gen

In [ ]:
%%writefile /content/mario_compare/optimizers/map_elites.py
"""
MAP-Elites (Mouret & Clune 2015) over flat MLP weights.

Instead of one population chasing one score, keep a GRID of niches, each
holding the best network found for one kind of behaviour. Here a niche is
"where did the episode end": a column of the level (final x, 64-pixel bins)
times a height band (final y, 32-pixel bins). "Died in the first pit",
"stuck on top of the tall pipe" and "reached the second Goomba group" are
different cells, and each keeps its own champion.

Each iteration: pick random champions from the grid, mutate them, score the
batch on shared seeds, and put each child in its cell if the cell is empty or
the child beats the current champion.

Known weakness with noisy fitness: a champion's score comes from one
evaluation and is never re-checked, so lucky networks can hold a cell. That is
exactly what the validation seeds will show.
"""

from __future__ import annotations
import numpy as np

from policies.mlp import MLPPolicy
from core.rollout import BudgetExhausted, evaluate, seed_set
from optimizers.ga import uniform_crossover

X_BIN, Y_BIN = 64, 32


def cell_of(res: dict) -> tuple[int, int]:
    return int(res["final_x"] // X_BIN), int(res["final_y"] // Y_BIN)


def run_map_elites(env, budget, log, batch=24, sigma=0.05, crossover=True,
                   n_seeds=3, seed=0):
    rng = np.random.default_rng(seed)
    policy = MLPPolicy(rng=rng, n_inputs=env.observation_space.shape[0])
    n = policy.n_params

    grid: dict[tuple[int, int], tuple[float, np.ndarray]] = {}   # cell -> (fitness, genome)
    it = 0

    try:
        while True:
            seeds = seed_set(rng, n_seeds)
            if not grid:
                # first batch: random networks, same init as every other method
                genomes = [0.1 * rng.standard_normal(n) for _ in range(batch)]
            else:
                cells = list(grid)
                pick = lambda: grid[cells[rng.integers(len(cells))]][1]
                genomes = []
                for _ in range(batch):
                    child = pick().copy()
                    if crossover:
                        child = uniform_crossover(rng, child, pick())
                    genomes.append(child + sigma * rng.standard_normal(n))

            fits = np.empty(batch)
            placed = 0
            for i, g in enumerate(genomes):
                policy.set_params(g)
                res = evaluate(env, policy, seeds, budget)
                fits[i] = res["fitness"]
                c = cell_of(res)
                if c not in grid or fits[i] > grid[c][0]:
                    grid[c] = (fits[i], g)
                    placed += 1

            it += 1
            champion = max(v[0] for v in grid.values())
            log(dict(method="mapelites", iteration=it, frames_used=budget.used,
                     fitness=fits.mean(), fitness_std=fits.std(),
                     fitness_min=fits.min(), fitness_max=champion))
            print(f"  [mapelites] it={it:3d} batch best={fits.max():7.1f} grid best={champion:7.1f} "
                  f"cells={len(grid):3d} placed={placed:2d}  budget={budget.fraction:.1%}")
    except BudgetExhausted:
        pass    # only the child whose evaluation ran out is lost; earlier ones were paid for and placed

    return max(grid.values(), key=lambda v: v[0])[1], it

In [ ]:
%%writefile /content/mario_compare/rl/sb3_agents.py
"""
PPO, A2C and DQN via stable-baselines3, made comparable with the evolution methods.

What is shared with every other method:
  * the same MarioTileEnv (same observation, sticky actions, termination rules)
  * the same reward: our x-progress reward, per step, instead of a per-episode total
  * the same frame budget: every emulator frame, including reset noops, is counted
  * the same network: 208 -> 18 (ReLU) -> 12, argmax at test time

What differs is the point of the comparison: RL learns from every step's reward
with gradients, while evolution only sees one number per episode.

Budget: SB3 counts agent steps, not frames. FrameCounter measures real frames
and a callback stops training the moment the frame budget is spent.

Scoring: after training, the policy's own weights (not a copy trained
separately) are exported into the flat MLPPolicy layout, so validation, test,
watch and score_val treat RL models exactly like evolved ones.
"""

from __future__ import annotations
import numpy as np
import gymnasium as gym
import torch.nn as nn
from gymnasium.wrappers import TimeLimit
from stable_baselines3 import PPO, A2C, DQN
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor

from policies.mlp import N_HIDDEN

MAX_STEPS = 3000        # same episode cap as core.rollout.rollout


class FrameCounter(gym.Wrapper):
    """Charges every emulator frame (steps and reset noops) to a FrameBudget."""

    def __init__(self, env, budget):
        super().__init__(env)
        self.budget = budget
        self._last = 0

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.budget.used += info["frames"]          # noop frames of this reset
        self._last = info["frames"]
        return obs, info

    def step(self, action):
        obs, r, term, trunc, info = self.env.step(action)
        self.budget.used += info["frames"] - self._last
        self._last = info["frames"]
        return obs, r, term, trunc, info


class StopOnBudget(BaseCallback):
    """Ends learn() as soon as the frame budget is used up; logs progress every `every` frames."""

    def __init__(self, budget, log, method, every=50_000):
        super().__init__()
        self.budget, self.log, self.method, self.every = budget, log, method, every
        self.ep_returns, self._next = [], every

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if "episode" in info:                    # added by SB3's Monitor wrapper
                self.ep_returns.append(info["episode"]["r"])
        if self.budget.used >= self._next and self.ep_returns:
            recent = np.array(self.ep_returns[-20:])
            self.log(dict(method=self.method, iteration=self.num_timesteps,
                          frames_used=self.budget.used, fitness=recent.mean(),
                          fitness_std=recent.std(), fitness_min=recent.min(),
                          fitness_max=recent.max()))
            print(f"  [{self.method}] steps={self.num_timesteps:7d} "
                  f"recent episode return={recent.mean():7.1f}  "
                  f"budget={self.budget.fraction:.1%}")
            self._next += self.every
        return self.budget.used < self.budget.total


# Per-algorithm defaults (SB3 / RL-zoo style). Every one can be overridden
# from that algorithm's runner: run_ppo.py, run_a2c.py, run_dqn.py.
DEFAULTS = {
    "ppo": dict(learning_rate=3e-4, n_steps=512, batch_size=64, n_epochs=4,
                gamma=0.99, gae_lambda=0.95, clip_range=0.2, ent_coef=0.01),
    "a2c": dict(learning_rate=7e-4, n_steps=16, gamma=0.99, gae_lambda=1.0, ent_coef=0.01),
    "dqn": dict(learning_rate=1e-4, buffer_size=100_000, learning_starts=5_000,
                batch_size=64, gamma=0.99, train_freq=4, target_update_interval=2_000,
                exploration_fraction=0.3, exploration_final_eps=0.05),
}
ALGOS = {"ppo": PPO, "a2c": A2C, "dqn": DQN}


def make_model(algo: str, env, seed: int, **overrides):
    """Same 18-unit ReLU policy network for every algorithm."""
    hp = {**DEFAULTS[algo], **{k: v for k, v in overrides.items() if v is not None}}
    if algo == "dqn":
        net = dict(net_arch=[N_HIDDEN], activation_fn=nn.ReLU)
    else:
        # actor gets the benchmark network; the critic (value head) is only a
        # training aid, is never used to act, and gets a normal-size network
        net = dict(net_arch=dict(pi=[N_HIDDEN], vf=[64, 64]), activation_fn=nn.ReLU)
    return ALGOS[algo]("MlpPolicy", env, seed=seed, verbose=0, policy_kwargs=net, **hp)


def export_to_mlp(model, algo: str) -> np.ndarray:
    """SB3 policy -> flat vector in MLPPolicy's layout [W1, b1, W2, b2].

    PPO/A2C: argmax over action logits. DQN: argmax over Q-values. Both are
    Linear -> ReLU -> Linear -> argmax, exactly what MLPPolicy.act computes.
    """
    if algo in ("ppo", "a2c"):
        hidden = model.policy.mlp_extractor.policy_net[0]
        out = model.policy.action_net
    else:
        hidden = model.q_net.q_net[0]
        out = model.q_net.q_net[2]
    t = lambda p: p.detach().cpu().numpy().astype(np.float64)
    return np.concatenate([t(hidden.weight).T.ravel(), t(hidden.bias),
                           t(out.weight).T.ravel(), t(out.bias)])


def run_rl(algo: str, env, budget, log, seed=0, **hp):
    wrapped = Monitor(TimeLimit(FrameCounter(env, budget), MAX_STEPS))
    total_steps = budget.total // env.frame_skip      # upper bound; the callback stops on frames
    model = make_model(algo, wrapped, seed, **hp)
    model.learn(total_timesteps=total_steps, callback=StopOnBudget(budget, log, algo))
    return export_to_mlp(model, algo), model.num_timesteps, model

In [ ]:
%%writefile /content/mario_compare/experiments/common.py
"""
Shared plumbing for the per-algorithm runners (run_hc.py, run_cem.py, ...).

Every runner does the same three things, and they live here so no algorithm can
quietly do them differently:

  setup()   -- same env, same budget, one CSV + weights file per run, with every
               setting in the file name
  finish()  -- score the final weights on VAL_SEEDS (for choosing settings) and
               TEST_SEEDS (for the final report), log both, save the weights

(run_ga.py / run_es.py predate this file and do the same steps inline.)
"""

from __future__ import annotations
import argparse, time
from pathlib import Path

import numpy as np

from envs.mario_env import MarioTileEnv, OBS_MODES
from policies.mlp import MLPPolicy
from core.rollout import FrameBudget, evaluate, VAL_SEEDS, TEST_SEEDS
from experiments.run_baselines import log_row, RESULTS


def base_parser(description: str) -> argparse.ArgumentParser:
    ap = argparse.ArgumentParser(description=description)
    ap.add_argument("--budget", type=int, default=1_000_000, help="emulator frames")
    ap.add_argument("--seed", type=int, default=0, help="run seed")
    ap.add_argument("--n-seeds", type=int, default=3, help="episodes per fitness evaluation")
    ap.add_argument("--obs", choices=OBS_MODES, default="grid")
    ap.add_argument("--level", default="SuperMarioBros-1-1-v0")
    ap.add_argument("--sticky-prob", type=float, default=0.2)
    return ap


def setup(method: str, args, settings: str):
    """Returns (env, budget, log, tag). `settings` is this algorithm's part of the name."""
    env = MarioTileEnv(level=args.level, sticky_prob=args.sticky_prob, obs_mode=args.obs)
    tag = (f"{time.strftime('%Y%m%d-%H%M%S')}_seed{args.seed}_sticky{args.sticky_prob}"
           f"_{settings}_ns{args.n_seeds}_b{args.budget // 1000}k"
           f"{'' if args.obs == 'grid' else '_obs' + args.obs}")
    out = RESULTS / f"{method}_{tag}.csv"

    def log(row: dict) -> None:
        log_row(out, dict(row, sticky_prob=env.sticky_prob), {})

    log.out = out
    return env, FrameBudget(args.budget), log, tag


def finish(method: str, env, params: np.ndarray, iterations: int, budget, log, tag: str):
    """Validation + held-out scores for the final weights, then save them."""
    policy = MLPPolicy(n_inputs=env.observation_space.shape[0])
    policy.set_params(params)
    val = evaluate(env, policy, VAL_SEEDS)
    test = evaluate(env, policy, TEST_SEEDS)
    print(f"  [{method}] {iterations} iterations, {budget.used:,} frames")
    print(f"  [{method}] validation: fitness={val['fitness']:.1f} x={val['max_x']:.1f}")
    print(f"  [{method}] held-out:   fitness={test['fitness']:.1f} x={test['max_x']:.1f}")
    log(dict(method=f"{method}_val", iteration=iterations, frames_used=budget.used, **val))
    log(dict(method=f"{method}_heldout", iteration=iterations, frames_used=budget.used, **test))
    np.save(RESULTS / f"{method}_best_{tag}.npy", params)
    env.close()
    print(f"wrote {log.out}")

In [ ]:
%%writefile /content/mario_compare/experiments/run_baselines.py
"""
Step 3 experiment: establish the floors.

Run this before any optimizer. It produces the two reference numbers that make
the rest of the study interpretable:

  1. HOLD-RIGHT   -- what a 3-line script achieves with no learning at all.
                     On 1-1 this is embarrassingly good. Any evolved policy
                     that does not clearly beat it has learned nothing.

  2. RANDOM SEARCH -- sample MLP weight vectors, keep the best. This is the
                     GA with selection, crossover and mutation removed. The
                     gap between GA and this is the only honest measure of
                     what the genetic machinery actually contributes.

Usage:
    python -m experiments.run_baselines --budget 200000
"""

from __future__ import annotations
import argparse, csv, sys, time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from envs.mario_env import MarioTileEnv, OBS_MODES
from policies.mlp import MLPPolicy, HoldRightPolicy, RandomPolicy
from core.rollout import FrameBudget, BudgetExhausted, evaluate, seed_set, TEST_SEEDS, VAL_SEEDS

RESULTS = Path(__file__).resolve().parents[1] / "results"
RESULTS.mkdir(exist_ok=True)


# Fixed column order. Scripted and search rows carry different keys, so letting
# DictWriter take its columns from whichever row comes first misaligns the file.
FIELDS = ["method", "iteration", "frames_used", "sticky_prob",
          "fitness", "fitness_std", "fitness_min", "fitness_max",
          "max_x", "best_x", "flag_rate", "env_reward", "frames", "wall_s"]


def log_row(path: Path, row: dict, header_written: dict) -> None:
    new = not path.exists()
    with path.open("a", newline="") as f:
        # extra keys (e.g. evaluate()'s final_x/final_y) are not columns: ignore them
        w = csv.DictWriter(f, fieldnames=FIELDS, restval="", extrasaction="ignore")
        if new and not header_written.get(path):
            w.writeheader()
            header_written[path] = True
        w.writerow(row)


def run_scripted(env, name, policy, seeds, out):
    t0 = time.time()
    res = evaluate(env, policy, seeds)
    res.update(method=name, iteration=0, frames_used=res["frames"],
               wall_s=round(time.time() - t0, 1), sticky_prob=env.sticky_prob)
    log_row(out, res, {})
    print(f"{name:22s} fitness={res['fitness']:8.1f} "
          f"x={res['max_x']:7.1f} best_x={res['best_x']:5d} "
          f"flag={res['flag_rate']:.2f}  ({res['frames']} frames)")
    return res


def run_random_search(env, budget, out, best_path, n_seeds=3, seed=0):
    """Control condition for the GA."""
    rng = np.random.default_rng(seed)
    policy = MLPPolicy(rng=rng, n_inputs=env.observation_space.shape[0])
    best_fit, best_params, it = -np.inf, policy.get_params(), 0

    try:
        while True:
            policy.set_params(0.1 * rng.standard_normal(policy.n_params))
            res = evaluate(env, policy, seed_set(rng, n_seeds), budget)
            it += 1
            if res["fitness"] > best_fit:
                best_fit, best_params = res["fitness"], policy.get_params()
                print(f"  [random-search] it={it:4d} new best={best_fit:8.1f} "
                      f"x={res['max_x']:7.1f}  budget={budget.fraction:.1%}")
            log_row(out, dict(method="random_search", iteration=it,
                              frames_used=budget.used, **res,
                              sticky_prob=env.sticky_prob), {})
    except BudgetExhausted:
        pass

    np.save(best_path, best_params)
    print(f"  [random-search] {it} candidates evaluated, best={best_fit:.1f}")

    policy.set_params(best_params)
    val = evaluate(env, policy, VAL_SEEDS)
    print(f"  [random-search] validation: fitness={val['fitness']:.1f} x={val['max_x']:.1f}")
    log_row(out, dict(method="random_search_val", iteration=it,
                      frames_used=budget.used, **val,
                      sticky_prob=env.sticky_prob), {})
    final = evaluate(env, policy, TEST_SEEDS)
    print(f"  [random-search] held-out: fitness={final['fitness']:.1f} "
          f"x={final['max_x']:.1f}")
    log_row(out, dict(method="random_search_heldout", iteration=it,
                      frames_used=budget.used, **final,
                      sticky_prob=env.sticky_prob), {})
    return final


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--budget", type=int, default=200_000,
                    help="emulator frames for random search")
    ap.add_argument("--level", default="SuperMarioBros-1-1-v0")
    ap.add_argument("--frame-skip", type=int, default=4)
    ap.add_argument("--sticky-prob", type=float, default=0.2)
    ap.add_argument("--seed", type=int, default=0, help="random-search RNG seed")
    ap.add_argument("--obs", choices=OBS_MODES, default="grid")
    args = ap.parse_args()

    env = MarioTileEnv(level=args.level, frame_skip=args.frame_skip,
                       sticky_prob=args.sticky_prob, obs_mode=args.obs)
    # one file per run, so rows from different settings never mix
    tag = (f"{time.strftime('%Y%m%d-%H%M%S')}_seed{args.seed}_sticky{args.sticky_prob}"
           f"{'' if args.obs == 'grid' else '_obs' + args.obs}")
    out = RESULTS / f"baselines_{tag}.csv"

    print(f"\n=== scripted baselines ({len(TEST_SEEDS)} held-out seeds) ===")
    run_scripted(env, "random_actions", RandomPolicy(0), TEST_SEEDS, out)
    run_scripted(env, "hold_right", HoldRightPolicy(jump_period=0), TEST_SEEDS, out)
    for period in (12, 20, 30, 45):
        run_scripted(env, f"hold_right_jump{period}",
                     HoldRightPolicy(jump_period=period), TEST_SEEDS, out)

    print(f"\n=== random weight search (budget {args.budget:,} frames) ===")
    run_random_search(env, FrameBudget(args.budget), out,
                      RESULTS / f"random_search_best_{tag}.npy", seed=args.seed)

    env.close()
    print(f"\nwrote {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/mario_compare/experiments/run_ga.py
"""
Run the GA once.

Usage:
    python -m experiments.run_ga --budget 200000 --seed 0
"""

from __future__ import annotations
import argparse, sys, time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from envs.mario_env import MarioTileEnv, OBS_MODES
from core.rollout import FrameBudget, evaluate, VAL_SEEDS
from optimizers.ga import run_ga
from policies.mlp import MLPPolicy
from experiments.run_baselines import log_row, RESULTS


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--budget", type=int, default=1_000_000)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--pop", type=int, default=24)
    ap.add_argument("--sigma", type=float, default=0.03)
    ap.add_argument("--crossover", action="store_true")
    ap.add_argument("--obs", choices=OBS_MODES, default="grid",
                    help="network input: grid | act | prev | both (see envs/mario_env.py)")
    ap.add_argument("--n-seeds", type=int, default=3, help="episodes per fitness evaluation")
    ap.add_argument("--stall-gens", type=int, default=0,
                    help="raise sigma after N gens without progress (0 = off)")
    ap.add_argument("--level", default="SuperMarioBros-1-1-v0")
    ap.add_argument("--sticky-prob", type=float, default=0.2)
    args = ap.parse_args()

    env = MarioTileEnv(level=args.level, sticky_prob=args.sticky_prob, obs_mode=args.obs)
    # every setting in the name, so parallel runs started in the same second never share a file
    tag = (f"{time.strftime('%Y%m%d-%H%M%S')}_seed{args.seed}_sticky{args.sticky_prob}"
           f"_pop{args.pop}_sig{args.sigma}_ns{args.n_seeds}_b{args.budget // 1000}k"
           f"{'_xover' if args.crossover else ''}"
           f"{'_st' + str(args.stall_gens) if args.stall_gens else ''}"
           f"{'' if args.obs == 'grid' else '_obs' + args.obs}")
    out = RESULTS / f"ga_{tag}.csv"
    log = lambda row: log_row(out, dict(row, sticky_prob=env.sticky_prob), {})

    budget = FrameBudget(args.budget)
    best, final, gen = run_ga(env, budget, log, pop_size=args.pop,
                              sigma=args.sigma, crossover=args.crossover,
                              n_seeds=args.n_seeds, seed=args.seed,
                              stall_gens=args.stall_gens)

    # validation score: use THIS to compare settings. ga_heldout is for the final report only.
    policy = MLPPolicy(n_inputs=env.observation_space.shape[0])
    policy.set_params(best)
    val = evaluate(env, policy, VAL_SEEDS)
    print(f"  [ga] validation: fitness={val['fitness']:.1f} x={val['max_x']:.1f}")
    log(dict(method="ga_val", iteration=gen, frames_used=budget.used, **val))
    log(dict(method="ga_heldout", iteration=gen, frames_used=budget.used, **final))
    np.save(RESULTS / f"ga_best_{tag}.npy", best)
    env.close()
    print(f"wrote {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/mario_compare/experiments/run_es.py
"""
Run OpenAI-style ES once.

Usage:
    python -m experiments.run_es --budget 1000000 --seed 0
"""

from __future__ import annotations
import argparse, sys, time
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from envs.mario_env import MarioTileEnv, OBS_MODES
from core.rollout import FrameBudget, evaluate, VAL_SEEDS
from optimizers.es import run_es
from policies.mlp import MLPPolicy
from experiments.run_baselines import log_row, RESULTS


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--budget", type=int, default=1_000_000)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--pairs", type=int, default=12, help="antithetic pairs per iteration")
    ap.add_argument("--sigma", type=float, default=0.02, help="perturbation size")
    ap.add_argument("--lr", type=float, default=0.01, help="Adam learning rate")
    ap.add_argument("--n-seeds", type=int, default=3, help="episodes per fitness evaluation")
    ap.add_argument("--obs", choices=OBS_MODES, default="grid")
    ap.add_argument("--level", default="SuperMarioBros-1-1-v0")
    ap.add_argument("--sticky-prob", type=float, default=0.2)
    args = ap.parse_args()

    env = MarioTileEnv(level=args.level, sticky_prob=args.sticky_prob, obs_mode=args.obs)
    tag = (f"{time.strftime('%Y%m%d-%H%M%S')}_seed{args.seed}_sticky{args.sticky_prob}"
           f"_pairs{args.pairs}_sig{args.sigma}_lr{args.lr}_ns{args.n_seeds}"
           f"_b{args.budget // 1000}k"
           f"{'' if args.obs == 'grid' else '_obs' + args.obs}")
    out = RESULTS / f"es_{tag}.csv"
    log = lambda row: log_row(out, dict(row, sticky_prob=env.sticky_prob), {})

    budget = FrameBudget(args.budget)
    theta, final, it = run_es(env, budget, log, n_pairs=args.pairs, sigma=args.sigma,
                              lr=args.lr, n_seeds=args.n_seeds, seed=args.seed)

    # validation score: use THIS to compare settings. es_heldout is for the final report only.
    policy = MLPPolicy(n_inputs=env.observation_space.shape[0])
    policy.set_params(theta)
    val = evaluate(env, policy, VAL_SEEDS)
    print(f"  [es] validation: fitness={val['fitness']:.1f} x={val['max_x']:.1f}")
    log(dict(method="es_val", iteration=it, frames_used=budget.used, **val))
    log(dict(method="es_heldout", iteration=it, frames_used=budget.used, **final))
    np.save(RESULTS / f"es_best_{tag}.npy", theta)
    env.close()
    print(f"wrote {out}")


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/mario_compare/experiments/run_hc.py
"""
Run the (1+1) hill climber once.

Usage:
    python -m experiments.run_hc --budget 1000000 --seed 0
"""

from __future__ import annotations
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from experiments.common import base_parser, setup, finish
from optimizers.hill_climber import run_hill_climber


def main():
    ap = base_parser("(1+1) hill climber")
    ap.add_argument("--sigma", type=float, default=0.05, help="mutation size")
    args = ap.parse_args()

    env, budget, log, tag = setup("hc", args, f"sig{args.sigma}")
    params, it = run_hill_climber(env, budget, log, sigma=args.sigma,
                                  n_seeds=args.n_seeds, seed=args.seed)
    finish("hc", env, params, it, budget, log, tag)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/mario_compare/experiments/run_cem.py
"""
Run the cross-entropy method once.

Usage:
    python -m experiments.run_cem --budget 1000000 --seed 0
"""

from __future__ import annotations
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from experiments.common import base_parser, setup, finish
from optimizers.cem import run_cem


def main():
    ap = base_parser("cross-entropy method")
    ap.add_argument("--pop", type=int, default=24)
    ap.add_argument("--elite-frac", type=float, default=0.2, help="fraction refit each iteration")
    ap.add_argument("--extra-std", type=float, default=0.02, help="noise floor on the std")
    args = ap.parse_args()

    env, budget, log, tag = setup(
        "cem", args, f"pop{args.pop}_elite{args.elite_frac}_xstd{args.extra_std}")
    params, it = run_cem(env, budget, log, pop_size=args.pop, elite_frac=args.elite_frac,
                         extra_std=args.extra_std, n_seeds=args.n_seeds, seed=args.seed)
    finish("cem", env, params, it, budget, log, tag)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/mario_compare/experiments/run_cmaes.py
"""
Run CMA-ES once (sep-CMA-ES, diagonal covariance, unless --full).

Usage:
    python -m experiments.run_cmaes --budget 1000000 --seed 0
"""

from __future__ import annotations
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from experiments.common import base_parser, setup, finish
from optimizers.cmaes import run_cmaes


def main():
    ap = base_parser("CMA-ES")
    ap.add_argument("--pop", type=int, default=24)
    ap.add_argument("--sigma0", type=float, default=0.1, help="initial step size")
    ap.add_argument("--full", action="store_true", help="full covariance (slow at 3990 dims)")
    args = ap.parse_args()

    env, budget, log, tag = setup(
        "cmaes", args, f"pop{args.pop}_sig{args.sigma0}{'_full' if args.full else ''}")
    params, it = run_cmaes(env, budget, log, pop_size=args.pop, sigma0=args.sigma0,
                           diagonal=not args.full, n_seeds=args.n_seeds, seed=args.seed)
    finish("cmaes", env, params, it, budget, log, tag)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/mario_compare/experiments/run_novelty.py
"""
Run novelty search (GA + novelty/fitness blend) once.

Usage:
    python -m experiments.run_novelty --budget 1000000 --seed 0
    python -m experiments.run_novelty --novelty-weight 1.0     # pure novelty search
"""

from __future__ import annotations
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from experiments.common import base_parser, setup, finish
from optimizers.novelty import run_novelty


def main():
    ap = base_parser("novelty search")
    ap.add_argument("--pop", type=int, default=24)
    ap.add_argument("--sigma", type=float, default=0.05)
    ap.add_argument("--no-crossover", action="store_true")
    ap.add_argument("--novelty-weight", type=float, default=0.5,
                    help="1 = pure novelty, 0 = plain GA")
    args = ap.parse_args()

    env, budget, log, tag = setup(
        "novelty", args, f"pop{args.pop}_sig{args.sigma}_nw{args.novelty_weight}"
                         f"{'' if args.no_crossover else '_xover'}")
    params, it = run_novelty(env, budget, log, pop_size=args.pop, sigma=args.sigma,
                             crossover=not args.no_crossover,
                             novelty_weight=args.novelty_weight,
                             n_seeds=args.n_seeds, seed=args.seed)
    finish("novelty", env, params, it, budget, log, tag)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/mario_compare/experiments/run_mapelites.py
"""
Run MAP-Elites once.

Usage:
    python -m experiments.run_mapelites --budget 1000000 --seed 0
"""

from __future__ import annotations
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from experiments.common import base_parser, setup, finish
from optimizers.map_elites import run_map_elites


def main():
    ap = base_parser("MAP-Elites")
    ap.add_argument("--batch", type=int, default=24, help="children per iteration")
    ap.add_argument("--sigma", type=float, default=0.05)
    ap.add_argument("--no-crossover", action="store_true")
    args = ap.parse_args()

    env, budget, log, tag = setup(
        "mapelites", args, f"batch{args.batch}_sig{args.sigma}"
                           f"{'' if args.no_crossover else '_xover'}")
    params, it = run_map_elites(env, budget, log, batch=args.batch, sigma=args.sigma,
                                crossover=not args.no_crossover,
                                n_seeds=args.n_seeds, seed=args.seed)
    finish("mapelites", env, params, it, budget, log, tag)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/mario_compare/experiments/run_ppo.py
"""
Run PPO (stable-baselines3) once, on the same frame budget as every other method.

Usage:
    python -m experiments.run_ppo --budget 1000000 --seed 0
    python -m experiments.run_ppo --lr 1e-4 --ent-coef 0.02

--n-seeds is accepted for a uniform command line but unused: RL learns from
every step of every episode instead of scoring candidates on seed sets.
"""

from __future__ import annotations
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from experiments.common import base_parser, setup, finish
from rl.sb3_agents import run_rl, DEFAULTS

D = DEFAULTS["ppo"]


def main():
    ap = base_parser("PPO")
    ap.add_argument("--lr", type=float, default=D["learning_rate"])
    ap.add_argument("--n-steps", type=int, default=D["n_steps"], help="steps collected per update")
    ap.add_argument("--batch-size", type=int, default=D["batch_size"])
    ap.add_argument("--n-epochs", type=int, default=D["n_epochs"], help="passes over each rollout")
    ap.add_argument("--clip", type=float, default=D["clip_range"], help="PPO clip range")
    ap.add_argument("--ent-coef", type=float, default=D["ent_coef"], help="entropy bonus")
    args = ap.parse_args()

    env, budget, log, tag = setup(
        "ppo", args, f"lr{args.lr}_nst{args.n_steps}_ent{args.ent_coef}_clip{args.clip}")
    params, steps, _ = run_rl("ppo", env, budget, log, seed=args.seed,
                              learning_rate=args.lr, n_steps=args.n_steps,
                              batch_size=args.batch_size, n_epochs=args.n_epochs,
                              clip_range=args.clip, ent_coef=args.ent_coef)
    finish("ppo", env, params, steps, budget, log, tag)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/mario_compare/experiments/run_a2c.py
"""
Run A2C (stable-baselines3) once, on the same frame budget as every other method.

Usage:
    python -m experiments.run_a2c --budget 1000000 --seed 0

--n-seeds is accepted for a uniform command line but unused: RL learns from
every step of every episode instead of scoring candidates on seed sets.
"""

from __future__ import annotations
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from experiments.common import base_parser, setup, finish
from rl.sb3_agents import run_rl, DEFAULTS

D = DEFAULTS["a2c"]


def main():
    ap = base_parser("A2C")
    ap.add_argument("--lr", type=float, default=D["learning_rate"])
    ap.add_argument("--n-steps", type=int, default=D["n_steps"], help="steps per update")
    ap.add_argument("--ent-coef", type=float, default=D["ent_coef"], help="entropy bonus")
    args = ap.parse_args()

    env, budget, log, tag = setup("a2c", args, f"lr{args.lr}_nst{args.n_steps}_ent{args.ent_coef}")
    params, steps, _ = run_rl("a2c", env, budget, log, seed=args.seed,
                              learning_rate=args.lr, n_steps=args.n_steps,
                              ent_coef=args.ent_coef)
    finish("a2c", env, params, steps, budget, log, tag)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/mario_compare/experiments/run_dqn.py
"""
Run DQN (stable-baselines3) once, on the same frame budget as every other method.

Usage:
    python -m experiments.run_dqn --budget 1000000 --seed 0

--n-seeds is accepted for a uniform command line but unused: RL learns from
every step of every episode instead of scoring candidates on seed sets.
"""

from __future__ import annotations
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from experiments.common import base_parser, setup, finish
from rl.sb3_agents import run_rl, DEFAULTS

D = DEFAULTS["dqn"]


def main():
    ap = base_parser("DQN")
    ap.add_argument("--lr", type=float, default=D["learning_rate"])
    ap.add_argument("--buffer", type=int, default=D["buffer_size"], help="replay buffer size (steps)")
    ap.add_argument("--learning-starts", type=int, default=D["learning_starts"],
                    help="random steps before training starts")
    ap.add_argument("--target-update", type=int, default=D["target_update_interval"],
                    help="steps between target-network syncs")
    ap.add_argument("--explore-frac", type=float, default=D["exploration_fraction"],
                    help="fraction of training over which epsilon decays 1 -> final")
    ap.add_argument("--final-eps", type=float, default=D["exploration_final_eps"])
    args = ap.parse_args()

    env, budget, log, tag = setup(
        "dqn", args, f"lr{args.lr}_buf{args.buffer // 1000}k_xf{args.explore_frac}")
    params, steps, _ = run_rl("dqn", env, budget, log, seed=args.seed,
                              learning_rate=args.lr, buffer_size=args.buffer,
                              learning_starts=args.learning_starts,
                              target_update_interval=args.target_update,
                              exploration_fraction=args.explore_frac,
                              exploration_final_eps=args.final_eps)
    finish("dqn", env, params, steps, budget, log, tag)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/mario_compare/experiments/watch.py
"""
Watch a policy play, in a window or saved to an .mp4.

Plays episode after episode (seed, seed+1, ...) until you press q, so you can
see where it succeeds and where it dies. Each episode ends the same way it does
in training: death, flag, or 240 frames without progress ("stuck").

Usage:
    python -m experiments.watch                                  # best GA model by held-out score
    python -m experiments.watch --policy es                      # best ES model
    python -m experiments.watch --policy ppo                     # ... or hc, cem, cmaes,
                                                                 #     novelty, mapelites, a2c, dqn
    python -m experiments.watch --weights results/ga_best_X.npy --seed 900003
    python -m experiments.watch --policy jump20                  # scripted baseline
    python -m experiments.watch --episodes 1                     # one episode, then close
    python -m experiments.watch --save mario.mp4 --episodes 5    # write video, no window

Keys in the window: q = quit, space = pause, n = skip to next episode.
"""

from __future__ import annotations
import argparse, csv, re, sys
from pathlib import Path

import cv2
import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from envs.mario_env import MarioTileEnv, obs_size
from policies.mlp import MLPPolicy, HoldRightPolicy

RESULTS = Path(__file__).resolve().parents[1] / "results"
SCALE = 3
END_HOLD_S = 2.0        # how long the last frame stays up with the end reason


def best_weights(method: str) -> Path:
    """The <method>_best_*.npy whose run has the highest held-out fitness."""
    scored = []
    for f in RESULTS.glob(f"{method}_*.csv"):
        npy = RESULTS / f.name.replace(f"{method}_", f"{method}_best_", 1).replace(".csv", ".npy")
        rows = [r for r in csv.DictReader(f.open()) if r["method"] == f"{method}_heldout"]
        if rows and npy.exists():
            scored.append((float(rows[-1]["fitness"]), npy))
    if not scored:
        sys.exit(f"no finished {method.upper()} run with saved weights in results/")
    fit, path = max(scored)
    print(f"best {method.upper()} model: {path.name}  (held-out fitness {fit:.1f})")
    return path


def obs_mode_of(path: Path) -> str:
    """Input type is in the file name (_obsact, _obsprev, _obsboth); none = grid."""
    m = re.search(r"_obs([a-z]+)", path.stem)
    return m.group(1) if m else "grid"


def load_policy(args):
    """Returns (policy, name, obs_mode)."""
    if args.policy == "jump20":
        return HoldRightPolicy(jump_period=20), "hold_right_jump20", "grid"
    path = Path(args.weights) if args.weights else best_weights(args.policy)
    mode = obs_mode_of(path)
    policy = MLPPolicy(n_inputs=obs_size(mode))
    policy.set_params(np.load(path))
    return policy, path.name, mode


def end_reason(info, term, trunc):
    if info.get("flag_get"):
        return "FLAG!"
    if term:
        return "died"
    if trunc:
        return "stuck (no progress)"
    return "quit"


def draw(env, lines):
    frame = cv2.cvtColor(env.env.unwrapped.screen, cv2.COLOR_RGB2BGR)
    frame = cv2.resize(frame, None, fx=SCALE, fy=SCALE,
                       interpolation=cv2.INTER_NEAREST)
    # dark panel so the overlay stays readable over the game's own HUD
    panel = frame[:12 + 32 * len(lines), :330]
    panel[:] = (panel * 0.35).astype(frame.dtype)
    for i, text in enumerate(lines):
        cv2.putText(frame, text, (10, 30 + 32 * i), cv2.FONT_HERSHEY_SIMPLEX,
                    0.8, (255, 255, 255), 2)
    return frame


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--weights", help=".npy from a GA / random-search run")
    ap.add_argument("--policy", default="ga",
                    choices=["ga", "es", "hc", "cem", "cmaes", "novelty", "mapelites",
                             "ppo", "a2c", "dqn", "jump20"],
                    help="which best model to load when --weights is not given")
    ap.add_argument("--seed", type=int, default=900_000, help="first episode's seed")
    ap.add_argument("--episodes", type=int, default=0, help="0 = until you press q")
    ap.add_argument("--sticky-prob", type=float, default=0.2)
    ap.add_argument("--fps", type=int, default=15, help="agent steps per second")
    ap.add_argument("--save", help="write an .mp4 here instead of opening a window")
    args = ap.parse_args()
    if args.save and args.episodes == 0:
        args.episodes = 5           # a video needs an end

    policy, name, mode = load_policy(args)
    env = MarioTileEnv(sticky_prob=args.sticky_prob, obs_mode=mode)
    title = f"Mario - {name}"
    writer = None

    def show(frame, wait_ms):
        """Returns the key pressed (window) or None (video)."""
        nonlocal writer
        if args.save:
            if writer is None:
                h, w = frame.shape[:2]
                writer = cv2.VideoWriter(args.save, cv2.VideoWriter_fourcc(*"mp4v"),
                                         args.fps, (w, h))
            for _ in range(max(1, wait_ms * args.fps // 1000)):
                writer.write(frame)
            return None
        cv2.imshow(title, frame)
        return cv2.waitKey(wait_ms) & 0xFF

    ep, quit_all, xs = 0, False, []
    while not quit_all and (args.episodes == 0 or ep < args.episodes):
        seed = args.seed + ep
        obs, info = env.reset(seed=seed)
        policy.reset()
        fitness, term, trunc, paused, skip = 0.0, False, False, False, False

        while not (term or trunc):
            if not paused:
                obs, r, term, trunc, info = env.step(policy.act(obs))
                fitness += r
            key = show(draw(env, [f"ep {ep + 1}  seed {seed}",
                                  f"x={info['max_x']}  fitness={fitness:.0f}"]),
                       1000 // args.fps)
            if key == ord("q"):
                quit_all = True
                break
            if key == ord("n"):
                skip = True
                break
            if key == ord(" "):
                paused = not paused

        reason = end_reason(info, term, trunc) if not skip else "skipped"
        xs.append(info["max_x"])
        print(f"ep {ep + 1:3d}  seed={seed}  x={info['max_x']:5d}  "
              f"fitness={fitness:7.1f}  {reason}")
        if not quit_all:
            key = show(draw(env, [f"ep {ep + 1}  seed {seed}",
                                  f"x={info['max_x']}  fitness={fitness:.0f}",
                                  f"END: {reason}"]),
                       int(END_HOLD_S * 1000))
            quit_all = key == ord("q")
        ep += 1

    print(f"{name}: {len(xs)} episodes, mean x={np.mean(xs):.1f}, best x={max(xs)}")
    if writer is not None:
        writer.release()
        print(f"wrote {args.save}")
    else:
        cv2.destroyAllWindows()
    env.close()


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/mario_compare/experiments/score_val.py
"""
Score every saved model on the validation seeds and compare settings.

Choose settings by the VAL column. The TEST column (the held-out score logged
at the end of each run) is shown only so you can see how much picking by test
flattered a setting -- do not choose by it.

Usage:
    python -m experiments.score_val              # all finished runs
    python -m experiments.score_val --workers 8
"""

from __future__ import annotations
import argparse, csv, re, sys
from collections import defaultdict
from multiprocessing import Pool
from pathlib import Path

import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parents[1]))

from envs.mario_env import MarioTileEnv, obs_size
from policies.mlp import MLPPolicy
from core.rollout import evaluate, VAL_SEEDS

RESULTS = Path(__file__).resolve().parents[1] / "results"

# methods whose runners use experiments/common.py: file names are
#   <method>_best_<time>_seed<N>_sticky<p>_<settings...>_ns<N>_b<N>k[_obs<mode>].npy
GENERIC = {"hc": "hill climber", "cem": "CEM", "cmaes": "CMA-ES", "novelty": "novelty",
           "mapelites": "MAP-Elites", "ppo": "PPO", "a2c": "A2C", "dqn": "DQN"}


def obs_mode_of(path: Path) -> str:
    m = re.search(r"_obs([a-z]+)", path.stem)
    return m.group(1) if m else "grid"


def heldout_row(csv_path: Path, method: str):
    if not csv_path.exists():
        return None
    rows = [r for r in csv.DictReader(csv_path.open()) if r["method"] == method]
    return rows[-1] if rows else None


def find_models():
    """[(setting label, run seed, npy path, test fitness)] for every finished run."""
    models = []
    for npy in RESULTS.glob("ga_best_*.npy"):
        row = heldout_row(RESULTS / npy.name.replace("ga_best_", "ga_", 1)
                          .replace(".npy", ".csv"), "ga_heldout")
        if row is None:
            continue
        name = npy.stem
        seed = int(re.search(r"_seed(\d+)", name).group(1))
        pop = re.search(r"_pop(\d+)", name)
        sig = re.search(r"_sig([\d.]+)", name)
        bud = re.search(r"_b(\d+)k", name)
        ns = re.search(r"_ns(\d+)", name)
        # runs from before settings were put in file names used the defaults
        label = " ".join(filter(None, [
            f"GA pop{pop.group(1) if pop else 24}",
            f"sig{sig.group(1) if sig else 0.03}",
            f"ns{ns.group(1)}" if ns and ns.group(1) != "3" else "",
            (f"{int(bud.group(1)) // 1000}M" if int(bud.group(1)) >= 1000 else f"{bud.group(1)}k")
            if bud else "1M",
            "+xover" if "_xover" in name else "",
            f"stall{re.search(r'_st(\d+)', name).group(1)}" if re.search(r"_st(\d+)", name) else "",
            "" if obs_mode_of(npy) == "grid" else f"obs={obs_mode_of(npy)}",
        ]))
        models.append((label, seed, npy, float(row["fitness"])))

    for npy in RESULTS.glob("es_best_*.npy"):
        row = heldout_row(RESULTS / npy.name.replace("es_best_", "es_", 1)
                          .replace(".npy", ".csv"), "es_heldout")
        if row is None:
            continue
        name = npy.stem
        get = lambda key: re.search(rf"_{key}([\d.]+?)(?:_|$)", name).group(1)
        bud = int(re.search(r"_b(\d+)k", name).group(1))
        label = " ".join(filter(None, [
            f"ES pairs{get('pairs')} sig{get('sig')} lr{get('lr')}",
            f"ns{get('ns')}" if get("ns") != "3" else "",
            f"{bud // 1000}M" if bud >= 1000 else f"{bud}k",
            "" if obs_mode_of(npy) == "grid" else f"obs={obs_mode_of(npy)}",
        ]))
        models.append((label, int(get("seed")), npy, float(row["fitness"])))

    for method, display in GENERIC.items():
        for npy in RESULTS.glob(f"{method}_best_*.npy"):
            row = heldout_row(RESULTS / npy.name.replace(f"{method}_best_", f"{method}_", 1)
                              .replace(".npy", ".csv"), f"{method}_heldout")
            if row is None:
                continue
            parts = npy.stem.split("_best_", 1)[1].split("_")[1:]    # drop the timestamp
            seed = int(parts[0][4:])
            settings = []
            for p in parts[2:]:                                    # after seed, sticky
                if p == "ns3":
                    continue
                m = re.fullmatch(r"b(\d+)k", p)
                if m:
                    k = int(m.group(1))
                    p = f"{k // 1000}M" if k >= 1000 else f"{k}k"
                elif p.startswith("obs"):
                    p = f"obs={p[3:]}"
                settings.append(p)
            models.append((f"{display} " + " ".join(settings), seed, npy, float(row["fitness"])))

    for npy in RESULTS.glob("random_search_best_*.npy"):
        row = heldout_row(RESULTS / npy.name.replace("random_search_best_", "baselines_", 1)
                          .replace(".npy", ".csv"), "random_search_heldout")
        if row is None:
            continue
        m = re.search(r"_seed(\d+)", npy.stem)
        frames = int(row["frames_used"])
        label = f"random search {round(frames / 1e6)}M" if frames >= 500_000 \
            else f"random search {frames // 1000}k"
        if obs_mode_of(npy) != "grid":
            label += f" obs={obs_mode_of(npy)}"
        models.append((label, int(m.group(1)) if m else 0, npy, float(row["fitness"])))
    return models


def score(npy: Path) -> float:
    mode = obs_mode_of(npy)
    env = MarioTileEnv(obs_mode=mode)
    policy = MLPPolicy(n_inputs=obs_size(mode))
    policy.set_params(np.load(npy))
    res = evaluate(env, policy, VAL_SEEDS)
    env.close()
    return res["fitness"]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--workers", type=int, default=8)
    args = ap.parse_args()

    models = find_models()
    print(f"scoring {len(models)} models on {len(VAL_SEEDS)} validation seeds ...")
    with Pool(args.workers) as pool:
        vals = pool.map(score, [m[2] for m in models])

    by_setting = defaultdict(list)
    for (label, seed, npy, test), val in zip(models, vals):
        by_setting[label].append((seed, val, test))

    print(f"\n{'setting':40s} {'runs':>4s} {'VAL mean ± std':>16s}   {'(test mean)':>11s}   per-run val")
    for label, runs in sorted(by_setting.items(), key=lambda kv: -np.mean([r[1] for r in kv[1]])):
        runs.sort()
        v = np.array([r[1] for r in runs])
        t = np.array([r[2] for r in runs])
        per_run = "  ".join(f"s{s}:{val:.0f}" for s, val, _ in runs)
        print(f"{label:40s} {len(runs):4d} {v.mean():8.1f} ± {v.std():5.1f}   ({t.mean():9.1f})   {per_run}")


if __name__ == "__main__":
    main()

## 3. Smoke test (~1 min)

Same command you ran on the Mac. It should print **held-out fitness=401.6**.
If it prints something else, the emulator or numpy behaves differently here —
tell me before trusting the big runs.

In [ ]:
!/content/venv/bin/python -u -m experiments.run_ga --budget 60000 --pop 8
# clear the smoke-test output so it is not counted in the benchmark
!rm -f results/*

## 4. Benchmark: GA × seeds and random search × seeds, in parallel

Edit `BUDGET` / `SEEDS` if you like. Colab usually has 2 CPU cores, so 6 runs
share 2 cores: expect roughly 3× longer than one run alone.
Progress prints every 60 s from the CSVs.

In [ ]:
import subprocess, time, glob, csv, os
BUDGET = 1_000_000
SEEDS  = [0, 1, 2]
PY = "/content/venv/bin/python"

procs = {}
for s in SEEDS:
    procs[f"ga{s}"] = subprocess.Popen(
        f"{PY} -u -m experiments.run_ga --budget {BUDGET} --seed {s} > results/log_ga{s}.txt 2>&1",
        shell=True)
    procs[f"rs{s}"] = subprocess.Popen(
        f"{PY} -u -m experiments.run_baselines --budget {BUDGET} --seed {s} > results/log_rs{s}.txt 2>&1",
        shell=True)
t0 = time.time()

def last_frames(pattern):
    files = sorted(glob.glob(pattern), key=os.path.getmtime)
    if not files: return 0
    rows = list(csv.DictReader(open(files[-1])))
    return min(int(rows[-1]["frames_used"]), BUDGET) if rows else 0

while any(p.poll() is None for p in procs.values()):
    time.sleep(60)
    status = []
    for s in SEEDS:
        status.append(f"ga{s} {last_frames(f'results/ga_*_seed{s}_*.csv')/BUDGET:4.0%}")
        status.append(f"rs{s} {last_frames(f'results/baselines_*_seed{s}_*.csv')/BUDGET:4.0%}")
    print(f"[{(time.time()-t0)/60:5.1f} min] " + "  ".join(status), flush=True)

print("done. exit codes:", {k: p.returncode for k, p in procs.items()})

## 5. Summary — paste this back

In [ ]:
import csv, glob, numpy as np

def heldout(pattern, method):
    out = []
    for f in sorted(glob.glob(pattern)):
        for r in csv.DictReader(open(f)):
            if r["method"] == method:
                out.append((f.split("/")[-1], float(r["fitness"]), float(r["max_x"]),
                            float(r["flag_rate"]), int(r["frames_used"])))
    return out

ga = heldout("results/ga_*.csv", "ga_heldout")
rs = heldout("results/baselines_*.csv", "random_search_heldout")
print(f"{'file':55s} {'fitness':>8s} {'max_x':>7s} {'flag':>5s} {'frames':>8s}")
for name, rows in (("GA", ga), ("random search", rs)):
    print(f"--- {name}")
    for f, fit, x, flag, fr in rows:
        print(f"{f:55s} {fit:8.1f} {x:7.1f} {flag:5.2f} {fr:8d}")
    if rows:
        fits = np.array([r[1] for r in rows])
        print(f"{'mean ± std over runs':55s} {fits.mean():8.1f} ± {fits.std():.1f}")

print("--- scripted (from one baselines file)")
f = sorted(glob.glob("results/baselines_*.csv"))[-1]
for r in csv.DictReader(open(f)):
    if r["method"] in ("random_actions", "hold_right", "hold_right_jump20"):
        print(f"{r['method']:55s} {float(r['fitness']):8.1f} {float(r['max_x']):7.1f}")

## 6. Watch the best GA run

In [ ]:
import glob, csv, subprocess
from IPython.display import HTML
from base64 import b64encode

# pick the GA run with the best held-out fitness
best = max(heldout("results/ga_*.csv", "ga_heldout"), key=lambda r: r[1])[0]
weights = "results/" + best.replace("ga_", "ga_best_").replace(".csv", ".npy")
print("weights:", weights)

!/content/venv/bin/python -m experiments.watch --weights {weights} --save /content/raw.mp4
# OpenCV writes mp4v, which browsers can't play; re-encode to H.264
!ffmpeg -y -loglevel error -i /content/raw.mp4 -vcodec libx264 -pix_fmt yuv420p /content/mario.mp4
HTML(f'<video width=600 controls><source src="data:video/mp4;base64,{b64encode(open("/content/mario.mp4","rb").read()).decode()}" type="video/mp4"></video>')

## 7. Download everything

In [ ]:
!cd /content && zip -qr mario_results.zip mario_compare/results mario.mp4
from google.colab import files
files.download("/content/mario_results.zip")